# Day 5

In [1]:
from dotenv import load_dotenv
load_dotenv()

from langchain_openai import ChatOpenAI
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langchain.agents import create_agent

# 1) 모델
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.5, max_tokens=2048)

# 2) 툴 (Wikipedia)
wiki_tool = WikipediaQueryRun(
    api_wrapper=WikipediaAPIWrapper(top_k_results=3, lang="en")
)
tools = [wiki_tool]

# 3) Agent 생성
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="너는 필요하면 tool을 사용할 수 있어."
)

# 4) 실행
result = agent.invoke({
    "messages": [
        {"role": "user", "content": "에드 시런은 몇 년도에 태어났니? 그리고 지금은 몇 살이니?"}
    ]
})

print(result["messages"][-1].content)

C:\Users\natur\AppData\Local\Temp\ipykernel_15960\727128321.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import WikipediaQueryRun


에드 시런(Ed Sheeran)은 1991년 2월 17일에 태어났습니다. 현재 2024년이므로, 그는 33세입니다.


# ChromaDB

In [2]:
import chromadb

client = chromadb.PersistentClient(path="my_chroma_db")
collection = client.get_or_create_collection(name="my_collection")

In [3]:
# 데이터 추가
collection.add(
    documents=["apple", "banana", "boat"],
    ids=["1", "2", "3"],
    metadatas=[{"type": "fruit"}, {"type": "fruit"}, {"type": "vehicle"}]
)

In [4]:
data = collection.get()
print(data)
data = collection.get(ids=['2'])
print(data)
data = collection.get(where={'type':'fruit'})
print(data)

{'ids': ['1', '2', '3'], 'embeddings': None, 'documents': ['apple', 'banana updated', 'boat'], 'uris': None, 'included': ['metadatas', 'documents'], 'data': None, 'metadatas': [{'type': 'fruit'}, {'type': 'fruit updated'}, {'type': 'vehicle'}]}
{'ids': ['2'], 'embeddings': None, 'documents': ['banana updated'], 'uris': None, 'included': ['metadatas', 'documents'], 'data': None, 'metadatas': [{'type': 'fruit updated'}]}
{'ids': ['1'], 'embeddings': None, 'documents': ['apple'], 'uris': None, 'included': ['metadatas', 'documents'], 'data': None, 'metadatas': [{'type': 'fruit'}]}


In [5]:
data = collection.query(query_texts=['boat'], n_results=2)
print(data)

{'ids': [['3', '1']], 'embeddings': None, 'documents': [['boat', 'apple']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[{'type': 'vehicle'}, {'type': 'fruit'}]], 'distances': [[0.0, 1.5457133054733276]]}


# update

In [6]:
collection.update(
    ids=["2"],
    documents=["banana updated"],
    metadatas=[{"type": "fruit updated"}]
)

# Delete

In [7]:
collection.delete(ids=["3"])

{'deleted': 1}

# DataLoader

In [8]:
import os
from langchain_community.document_loaders import TextLoader, PyPDFLoader, WebBaseLoader

os.environ["USER_AGENT"] = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3"

text_loader = TextLoader('data.txt', encoding='utf-8')
docs = text_loader.load()
print(docs)

pdf_loader = PyPDFLoader('data.pdf')
pdf_docs = pdf_loader.load()
print(pdf_docs)

web_loader = WebBaseLoader('https://en.wikipedia.org/wiki/Artificial_intelligence')
web_docs = web_loader.load()
print(web_docs)

USER_AGENT environment variable not set, consider setting it to identify your requests.


[Document(metadata={'source': 'data.txt'}, page_content='프랑스의 수도는 파리이다. 파리는 유럽에서 가장 인기 있는 관광 도시 중 하나로, 에펠탑과 루브르 박물관이 유명하다.\n\n독일의 수도는 베를린이다. 베를린은 역사적으로 중요한 도시이며, 베를린 장벽으로 유명하다.\n\n일본의 수도는 도쿄이다. 도쿄는 기술과 문화의 중심지로, 애니메이션과 음식 문화가 발달해 있다.\n\n영국의 수도는 런던이다. 런던은 전통과 현대가 공존하는 도시로, 빅벤과 타워 브리지가 유명하다.\n\n이탈리아의 수도는 로마이다. 로마는 고대 역사의 숨결이 살아 숨 쉬는 도시로, 콜로세움과 트레비 분수가 유명하다.\n\n미국의 수도는 워싱턴 D.C.이다. 워싱턴 D.C.는 세계 정치와 외교의 중심지로, 백악관과 링컨 기념관이 유명하다.\n\n호주의 수도는 캔버라이다. 캔버라는 계획적으로 조성된 행정 도시이며, 국회의사당과 아름다운 인공 호수인 벌리 그리핀 호수가 유명하다.\n\n스페인의 수도는 마드리드이다. 마드리드는 정열과 예술의 도시로, 프라도 미술관과 화려한 왕궁이 유명하다.')]
[Document(metadata={'producer': 'Hancom PDF 1.3.0.550', 'creator': 'Hwp 2024 13.0.0.3189', 'creationdate': '2026-06-19T14:02:13+09:00', 'author': 'grovy', 'moddate': '2026-06-19T14:02:13+09:00', 'pdfversion': '1.4', 'source': 'data.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}, page_content='프랑스의 수도는 파리이다. 파리는 유럽에서 가장 인기 있는 관광 도시 중 하나로, 에펠탑과 루브르 박물관이 유명하다.독일의 수도는 베를린이다. 베를린은 역사적으로 중요한 도시이며, 베를린 장벽으로 유명하다.일본의 수도는 도쿄이다. 

[Document(metadata={'source': 'https://en.wikipedia.org/wiki/Artificial_intelligence', 'title': 'Artificial intelligence - Wikipedia', 'language': 'en'}, page_content='\n\n\n\nArtificial intelligence - Wikipedia\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nJump to content\n\n\n\n\n\n\n\nMain menu\n\n\n\n\n\nMain menu\nmove to sidebar\nhide\n\n\n\n\t\tNavigation\n\t\n\n\nMain pageContentsCurrent eventsRandom articleAbout WikipediaContact us\n\n\n\n\n\n\t\tContribute\n\t\n\n\nHelpLearn to editCommunity portalRecent changesUpload fileSpecial pages\n\n\n\n\n\n\t\tPrint/export\n\t\n\n\nDownload as PDFPrintable version\n\n\n\n\n\n\t\tIn other projects\n\t\n\n\nAbstract WikipediaWikimedia CommonsWikibooksWikiquoteWikiversityWikidata item\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nSearch\n\n\n\n\n\n\n\n\n\n\n\nSearch\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nAppearance\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nDonate\n\nCreate account\n\nLog in\n\n\n\n\n\n\n\n\nPersonal tools\n\n\n\n\n\n\nDonate

In [9]:
text_loader = TextLoader('data.txt', encoding='utf-8')
docs = text_loader.load()
print(docs)

[Document(metadata={'source': 'data.txt'}, page_content='프랑스의 수도는 파리이다. 파리는 유럽에서 가장 인기 있는 관광 도시 중 하나로, 에펠탑과 루브르 박물관이 유명하다.\n\n독일의 수도는 베를린이다. 베를린은 역사적으로 중요한 도시이며, 베를린 장벽으로 유명하다.\n\n일본의 수도는 도쿄이다. 도쿄는 기술과 문화의 중심지로, 애니메이션과 음식 문화가 발달해 있다.\n\n영국의 수도는 런던이다. 런던은 전통과 현대가 공존하는 도시로, 빅벤과 타워 브리지가 유명하다.\n\n이탈리아의 수도는 로마이다. 로마는 고대 역사의 숨결이 살아 숨 쉬는 도시로, 콜로세움과 트레비 분수가 유명하다.\n\n미국의 수도는 워싱턴 D.C.이다. 워싱턴 D.C.는 세계 정치와 외교의 중심지로, 백악관과 링컨 기념관이 유명하다.\n\n호주의 수도는 캔버라이다. 캔버라는 계획적으로 조성된 행정 도시이며, 국회의사당과 아름다운 인공 호수인 벌리 그리핀 호수가 유명하다.\n\n스페인의 수도는 마드리드이다. 마드리드는 정열과 예술의 도시로, 프라도 미술관과 화려한 왕궁이 유명하다.')]


# Document Transformer

In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=50, chunk_overlap=10)

docs = splitter.split_documents(docs)
print(len(docs))
for doc in docs:
    print(doc.page_content)

16
프랑스의 수도는 파리이다. 파리는 유럽에서 가장 인기 있는 관광 도시 중 하나로, 에펠탑과
하나로, 에펠탑과 루브르 박물관이 유명하다.
독일의 수도는 베를린이다. 베를린은 역사적으로 중요한 도시이며, 베를린 장벽으로
베를린 장벽으로 유명하다.
일본의 수도는 도쿄이다. 도쿄는 기술과 문화의 중심지로, 애니메이션과 음식 문화가 발달해
문화가 발달해 있다.
영국의 수도는 런던이다. 런던은 전통과 현대가 공존하는 도시로, 빅벤과 타워 브리지가
타워 브리지가 유명하다.
이탈리아의 수도는 로마이다. 로마는 고대 역사의 숨결이 살아 숨 쉬는 도시로, 콜로세움과
콜로세움과 트레비 분수가 유명하다.
미국의 수도는 워싱턴 D.C.이다. 워싱턴 D.C.는 세계 정치와 외교의 중심지로,
외교의 중심지로, 백악관과 링컨 기념관이 유명하다.
호주의 수도는 캔버라이다. 캔버라는 계획적으로 조성된 행정 도시이며, 국회의사당과
국회의사당과 아름다운 인공 호수인 벌리 그리핀 호수가 유명하다.
스페인의 수도는 마드리드이다. 마드리드는 정열과 예술의 도시로, 프라도 미술관과 화려한
미술관과 화려한 왕궁이 유명하다.


In [11]:
from langchain_openai import OpenAIEmbeddings

embedding_model = OpenAIEmbeddings()

doc_texts = [doc.page_content for doc in docs]

vector = embedding_model.embed_documents(doc_texts)
print(len(vector))
print(vector)

16
[[0.0038600072730332613, 0.016181042417883873, 0.017420556396245956, -0.04823325201869011, -0.0026053364854305983, 0.04004168137907982, -0.02142203040421009, 0.008595758117735386, 0.0170028954744339, -0.018134623765945435, -0.007032893132418394, 0.015722962096333504, 0.017434028908610344, -0.026070207357406616, -0.014443029649555683, -0.01665259711444378, 0.008642913773655891, -0.005335298366844654, -4.499763235799037e-06, -0.0038903215900063515, -0.005406031385064125, -0.006015683524310589, 0.014523867517709732, 0.0037286458536982536, 0.026892058551311493, 0.019643597304821014, 0.004075575154274702, -0.01685469225049019, -0.02896689623594284, 0.005682227201759815, 0.00983527209609747, -0.0014096100348979235, -0.028023788705468178, -0.020654071122407913, 0.002499236958101392, 0.007255197037011385, -0.006830798462033272, -0.019347192719578743, -0.0027097521815449, -0.0032031999435275793, -0.004543760791420937, -0.02268848940730095, 0.01162044145166874, -0.01860617846250534, 0.0213411

# VectorDB에 저장하기

In [12]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embedding_model,
    collection_name="my_collection",
    persist_directory="./chroma_db"
)

# Retriever 와 LLM 생성과정

In [13]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")

retriever = vectorstore.as_retriever()

In [14]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

chat_prompt = ChatPromptTemplate.from_template(
    "다음 질문에 답해줘. 만약 문서에 없는 내용이면 지어내지 말고 없다고 답해줘.\n[참고문서]{context}\n질문: {question}"
)

def format_docs(docs):
    return "\n\n".join([doc.page_content for doc in docs])

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | chat_prompt
    | llm
    | StrOutputParser()
)

result = rag_chain.invoke("프랑스의 수도는?")
print(result)

프랑스의 수도는 파리입니다.


In [15]:
question = "에드 시런은 몇 년도에 태어났니? 그리고 지금은 몇 살이니?"
ret = rag_chain.invoke(question)
print(ret)

문서에 에드 시런에 대한 내용이 없으므로, 답변할 수 없습니다.


# 위키피디아 아이유 페이지를 가져오기
url = https://ko.wikipedia.org/wiki/%EC%95%84%EC%9D%B4%EC%9C%A0
#mw-content-text

In [16]:
import requests
from bs4 import BeautifulSoup

headers = {'user-agent' : 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3'}
resp = requests.get('https://en.wikipedia.org/wiki/Ed_Sheeran', headers=headers)
soup = BeautifulSoup(resp.text, 'html.parser')

div = soup.find('div', {'id': 'mw-content-text'})
print(div)

for tag in div.find_all(['p', 'h2', 'h3']):
    tag_text = tag.get_text()
    print(tag_text)
paras = div.find_all('p')
for para in paras:
    print(para)

    text = para.get_text()
    


<div class="mw-body-content" id="mw-content-text"><div class="mw-subjectpageheader">
</div><div class="mw-content-ltr mw-parser-output" dir="ltr" lang="en"><div class="shortdescription nomobile noexcerpt noprint searchaux" style="display:none">English singer-songwriter (born 1991)</div>
<p class="mw-empty-elt">
</p>
<style data-mw-deduplicate="TemplateStyles:r1358620234">@media(min-width:640px){.mw-parser-output .infobox{margin-left:1em;float:right;clear:right;width:22em}}.mw-parser-output .infobox-subbox{padding:0;border:none;margin:-3px;width:auto;min-width:100%;font-size:100%;clear:none;float:none;background-color:transparent;color:inherit}.mw-parser-output .infobox-3cols-child{margin:-3px}.mw-parser-output .infobox .navbar{font-size:100%}.mw-parser-output .infobox-hiddenrow,body.skin--responsive.skin--responsive .mw-parser-output .infobox .infobox-hiddenrow{display:none}@media screen{html.skin-theme-clientpref-night .mw-parser-output .infobox-full-data:not(.notheme)>div:not(.nothem

In [17]:
import requests
from bs4 import BeautifulSoup

headers = {'User-Agent': 'Mozilla/5.0'}
url = 'https://ko.wikipedia.org/wiki/%EC%95%84%EC%9D%B4%EC%9C%A0'
resp = requests.get(url, headers=headers)

soup= BeautifulSoup(resp.text, 'html.parser')

div = soup.find('div', {'id':'mw-content-text'})

for tag in div.find_all(['style', 'script', 'table', 'sup']):
    tag.decompose()

paras = div.find_all('p')

text_list = []
for p in paras:
    text = p.get_text().strip()
    if text:
        text_list.append(text)

for t in text_list:
    print(t)

아이유(IU, 본명: 이지은, 1993년 5월 16일~)는 대한민국의 싱어송라이터, 작곡가, 배우이다. 2008년, 첫 EP인 로스트 앤 파운드(Lost and Found)를 통해 가수로 데뷔했다.
아이유는 2008년 9월 18일 엠넷 M! Countdown에서 데뷔 싱글 "미아"로 데뷔하였고 2008년 9월 23일에 발매된 첫 번째 미니 음반 Lost and Found를 발매하였다. 데뷔 전부터 라디오에서 음악을 알렸던 아이유는 유영석, 유희열, 정재형 등의 많은 음악가로부터 호평을 받았다. 유영석은 "나이가 어린 소녀인데 노래를 너무 잘하고, 음악성이 뛰어난 것 같아서 이 앨범을 추천한다"라고 칭찬하였으며, 정재형은 "앞으로 많은 발전이  가수"라며 기대감을 보였다. 아이유는 음반이 발매되기 전인 2008년 9월 18일에 Mnet 《엠 카운트다운》에서 타이틀곡인 〈미아〉를 불러 첫 데뷔 무대를 가졌다. 타이틀곡 〈미아〉는 이종훈, 민웅식이 공동 작곡한 곡으로 일렉트로닉 사운드에 힙합 리듬이 섞인 하이브리드 팝 발라드이다. 앨범 발매 당시에는 별로 관심을 받지 못하였지만,〈미아〉를 통하여 아이유는 당시 16세의 소녀답지 않은 가창력으로 코러스와 아리아 부분까지 1인 3역을 소화해내며 조금씩 대중들에게 얼굴을 알렸다. 그 결과 2008년 10월에 문화체육관광부에서 추진하는 '이달의 우수 신인음반'에 아이유의 Lost and Found가 선정되었다.
아이유는 2009년 4월 23일에 정규 1집 Growing Up을 발매했다. 이 음반에는 발랄하고 밝은 느낌의 다양한 장르의 곡들이 수록되어 있다. 음반 발매 일주일 전인 2009년 4월 17일, KBS 《뮤직뱅크》에서 타이틀곡 〈Boo〉를 처음 선보였다. 깜찍한 무대 스타일로 이목을 집중시켰다. 아이유가 이름을 알리기 시작한 것은 이때부터인데, 2009년 5월부터 유행하기 시작한 'MR제거 동영상'에서 아이유는 빼어난 가창력을 뽐내 실시간 검색어에서 1위를 하는 등 대중들에게 이름을 알렸고, 가창력을 인정받았다. 

In [18]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
splitter.create_documents([text])


[Document(metadata={}, page_content='2026년, 앨범을 준비 중이다.')]

In [19]:
from langchain_openai import OpenAIEmbeddings

from langchain_community.vectorstores import Chroma

embedding_model = OpenAIEmbeddings()

vectorstore = Chroma.from_documents(

    documents=docs,

    embedding=embedding_model,

    persist_directory='my_Chroma_DB'

)

vectorstore.persist()






C:\Users\natur\AppData\Local\Temp\ipykernel_15960\4086862866.py:17: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vectorstore.persist()


In [20]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

prompt = ChatPromptTemplate.from_template(

    ''' 다음 문서를 참고해서 답하세요. 만약 문서에 없는 내용이면 지어내지 말고 없다고 말하세요

[참고문서]{context}

질문{question}

    '''
)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}

    | prompt

    | llm

    | StrOutputParser()

)

In [21]:
rag_chain.invoke("아이유 본명은 무엇인가요?")
print(result)

프랑스의 수도는 파리입니다.


# 야구 대표팀 RAG

In [22]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 1. 문서 로드
loader = TextLoader('soccer.txt', encoding='utf-8')
docs = loader.load()

# 2. 텍스트 분할
splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=30)
split_docs = splitter.split_documents(docs)
print(f"총 {len(split_docs)}개 청크로 분할")

# 3. ChromaDB에 저장
embedding_model = OpenAIEmbeddings()
print("새 ChromaDB 생성합니다 (야구 분석 중)...")
baseball_vectorstore = Chroma.from_documents(
    documents=split_docs,
    embedding=embedding_model,
    persist_directory='./baseball_chroma_db'
)
print("새 ChromaDB를 생성하고 로컬 디스크에 저장했습니다.")

# 4. RAG 체인 구성
llm = ChatOpenAI(model="gpt-4o-mini")
retriever = baseball_vectorstore.as_retriever(search_kwargs={"k": 5})

def format_docs(docs):
    return "\n\n".join([doc.page_content for doc in docs])

prompt = ChatPromptTemplate.from_template(
    "다음 문서를 참고해서 답하세요. 만약 문서에 없는 내용이면 지어내지 말고 없다고 말하세요.\n\n[참고문서]\n{context}\n\n질문: {question}"
)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# 5. 질문 테스트
query = "선발에서 제외된 선수 이름은?"
response = rag_chain.invoke(query)
print(f"\n[질문]: {query}")
print(f"[AI 답변]: {response}")

총 6개 청크로 분할
새 ChromaDB 생성합니다 (야구 분석 중)...


새 ChromaDB를 생성하고 로컬 디스크에 저장했습니다.



[질문]: 선발에서 제외된 선수 이름은?
[AI 답변]: 선발에서 제외된 선수는 김하성, 추신수, 류현진입니다.


In [23]:
# 질문 예시
response = rag_chain.invoke('선발에서 제외된 선수 이름은?')
print(response)

response = rag_chain.invoke('소집 선수는 몇 명이니?')
print(response)

response = rag_chain.invoke('내야수로 뽑힌 선수는 모두 몇 명이고, 이름이 뭐니?')
print(response)

선발에서 제외된 선수 이름은 김하성, 추신수, 류현진입니다.


소집 선수는 총 18명입니다. 외야수 4명, 지명타자 3명, 투수 11명을 합쳐서 4 + 3 + 11 = 18명입니다.


내야수로 뽑힌 선수는 7명입니다. 다음은 그들의 이름입니다:

1. 김혜성 (LA 다저스)
2. 박병호 (KT 위즈)
3. 오지환 (LG 트윈스)
4. 강백호 (KT 위즈)
5. 김도영 (KIA 타이거즈)
6. 노시환 (한화 이글스)
7. 최정 (SSG 랜더스)


# 키움 히어로즈 위키 크롤링 → RAG

In [ ]:
# 1. 키움 히어로즈 나무위키 크롤링
import requests
from bs4 import BeautifulSoup

headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'}
url = 'https://namu.wiki/w/%ED%82%A4%EC%9B%80%20%ED%9E%88%EC%96%B4%EB%A1%9C%EC%A6%88'
resp = requests.get(url, headers=headers)

soup = BeautifulSoup(resp.text, 'html.parser')

for tag in soup.find_all(['style', 'script', 'table', 'sup']):
    tag.decompose()

paras = soup.find_all(['p', 'div', 'span'])

text_list = []
for p in paras:
    text = p.get_text().strip()
    if text and len(text) > 20:
        text_list.append(text)

# 중복 제거
text_list = list(dict.fromkeys(text_list))

print(f"총 {len(text_list)}개 문단 크롤링 완료")
for t in text_list[:5]:
    print(t[:100])
    print("---")

In [25]:
# 2. 텍스트 분할 → 임베딩 → ChromaDB 저장
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
kiwoom_docs = splitter.create_documents(text_list)
print(f"총 {len(kiwoom_docs)}개 청크로 분할")

embedding_model = OpenAIEmbeddings()
print("새 ChromaDB 생성합니다 (키움 히어로즈 분석 중)...")
kiwoom_vectorstore = Chroma.from_documents(
    documents=kiwoom_docs,
    embedding=embedding_model,
    persist_directory='./kiwoom_chroma_db'
)
print("새 ChromaDB를 생성하고 로컬 디스크에 저장했습니다.")

총 13개 청크로 분할
새 ChromaDB 생성합니다 (키움 히어로즈 분석 중)...


새 ChromaDB를 생성하고 로컬 디스크에 저장했습니다.


In [26]:
# 3. RAG 체인 구성
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model="gpt-4o-mini")
retriever = kiwoom_vectorstore.as_retriever(search_kwargs={"k": 5})

def format_docs(docs):
    return "\n\n".join([doc.page_content for doc in docs])

prompt = ChatPromptTemplate.from_template(
    "다음 문서를 참고해서 답하세요. 만약 문서에 없는 내용이면 지어내지 말고 없다고 말하세요.\n\n[참고문서]\n{context}\n\n질문: {question}"
)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
import textwrap

# 4. 질문 예시
response = rag_chain.invoke('키움 히어로즈의 홈구장은 어디야?')
print(textwrap.fill(response, width=20))

response = rag_chain.invoke('키움 히어로즈의 전신 팀 이름은?')
print(textwrap.fill(response, width=20))

response = rag_chain.invoke('키움 히어로즈는 언제 창단했어?')
print(textwrap.fill(response, width=20))